In [1]:
! pip install bs4
! pip install lxml
! pip install html5lib


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [176]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import random

In [160]:
base_url = 'https://www.topcv.vn/tim-viec-lam-data-analyst-tai-ho-chi-minh-kl2?type_keyword=1&sba=1&locations=l2'

In [141]:
# Headers for request
HEADERS = ({
                'User-Agent' : 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/150.0.0.0 Safari/537.36', 
                'Accepted-Language': 'vi-VN,vi;q=0.9,en-US;q=0.8' })

In [161]:
page = requests.get(base_url, headers = HEADERS)

In [162]:
soup = BeautifulSoup(page.content, 'lxml')

In [163]:
job_links = soup.find_all('a', attrs={'aria-label': True})

In [184]:
link_list = []
for job in job_links:
    href = job.get('href')
    # Chỉ lấy link thật, bỏ qua javascript popup và None
    if href and href.startswith('https://www.topcv.vn/'):
        link_list.append(href)

data = []
for link in link_list:
    # Retry khi gặp lỗi 429 (Rate Limit)
    while True:
        new_webpage = requests.get(link, headers=HEADERS)
        if new_webpage.status_code == 429:
            print("Bị chặn 429, đợi lâu hơn rồi thử lại...")
            time.sleep(random.uniform(15, 30))
            continue
        break

    new_soup = BeautifulSoup(new_webpage.content, 'lxml')

    # Trích xuất Tên công việc
    job_title_element = new_soup.find('h1', class_='box-header-job__title')
    job_title = " ".join(job_title_element.text.split()) if job_title_element else ''

    # Trích xuất Tên công ty
    company_name_el = new_soup.find('a', class_='name')
    company_name = company_name_el.text.strip() if company_name_el else ''

    # Khởi tạo Dictionary theo đúng thứ tự bạn yêu cầu
    job_info = {
        'job_title': job_title,      # 1. Tên công việc
        'company_name': company_name, # 2. Tên công ty
        'link': link,                 # 3. Đường link
        'location': '',               # 4. Địa điểm
        'experience': '',             # 5. Kinh nghiệm
        'deadline': ''                # 6. Hạn ứng tuyển
    }

    # Bổ sung các thông tin chi tiết
    for item in new_soup.find_all('div', class_='list-info__content'):
        title_el = item.find('div', class_='list-info__content__title')
        desc_el = item.find('div', class_='list-info__content__desc')

        title = title_el.get_text(strip=True) if title_el else ''
        desc = desc_el.get_text(strip=True) if desc_el else ''

        if 'Địa điểm' in title:
            job_info['location'] = desc
        elif 'Kinh nghiệm' in title:
            job_info['experience'] = desc
        elif 'Hạn' in title:
            job_info['deadline'] = desc

    data.append(job_info)
    
    # Nghỉ 1 lần ngắn sau mỗi link thay vì nghỉ nhiều lần rải rác
    time.sleep(random.uniform(3, 7))

# In dữ liệu dạng dễ nhìn (Pretty Print)
import json
print(json.dumps(data, ensure_ascii=False, indent=4))

[
    {
        "job_title": "Data Analyst Ngân Hàng Shinhan",
        "company_name": "Ngân Hàng TNHH MTV Shinhan Việt Nam",
        "link": "https://www.topcv.vn/viec-lam/data-analyst-ngan-hang-shinhan/2174212.html?ta_source=JobSearchList_LinkDetail&u_sr_id=PkRogDac2NRCf3nmn5mk1q3ApbNYunwCQqlnDdeg_1785251203",
        "location": "Hồ Chí Minh",
        "experience": "1 năm",
        "deadline": "31/08/2026"
    },
    {
        "job_title": "Data Analyst Workforce Management_App Social Video",
        "company_name": "CÔNG TY TNHH TRANSCOSMOS VIỆT NAM",
        "link": "https://www.topcv.vn/brand/transcosmoshochiminh/tuyen-dung/data-analyst-workforce-management-app-social-video-j2238139.html?ta_source=JobSearchList_LinkDetail&u_sr_id=PkRogDac2NRCf3nmn5mk1q3ApbNYunwCQqlnDdeg_1785251203",
        "location": "Hồ Chí Minh",
        "experience": "1 năm",
        "deadline": "31/08/2026"
    },
    {
        "job_title": "Chuyên Viên Dữ Liệu / Data Executive / Bắt Buộc Biết Tiếng Trung /

In [185]:
# Lưu kết quả ra file JSON
with open("jobs.json", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=4)
print("Đã lưu dữ liệu vào jobs.json")

Đã lưu dữ liệu vào jobs.json


In [191]:
import pandas as pd

# Giả sử bạn đã thu thập xong list data ở trên
df = pd.DataFrame(data)

# Đổi lại tên cột hiển thị bằng tiếng Việt nếu muốn (Tùy chọn)
df = df.rename(columns={
    'job_title': 'Tên công việc',
    'company_name': 'Tên công ty',
    'link': 'Đường link',
    'location': 'Địa điểm',
    'experience': 'Kinh nghiệm',
    'deadline': 'Hạn ứng tuyển'
})

# Xuất ra file CSV (dùng utf-8-sig để xem tiếng Việt chuẩn trong Excel)
df.to_csv('topcv_jobs_pandas.csv', index=False, encoding='utf-8-sig')

print("Đã xuất file thành công!")

Đã xuất file thành công!


2/ Lấy đại diện để test code.

In [165]:
job_list = link_list[0]

In [166]:
job_list

'https://www.topcv.vn/viec-lam/data-analyst-ngan-hang-shinhan/2174212.html?ta_source=JobSearchList_LinkDetail&u_sr_id=PkRogDac2NRCf3nmn5mk1q3ApbNYunwCQqlnDdeg_1785251203'

In [167]:
new_webpage = requests.get(job_list, headers = HEADERS)

In [168]:
new_webpage

<Response [200]>

In [169]:
new_soup = BeautifulSoup(new_webpage.content, 'lxml')

In [170]:
# Tìm thẻ h1 chứa tên công việc
job_title_element = new_soup.find('h1', class_='box-header-job__title')

# Lấy toàn bộ text, dùng strip=True để loại bỏ khoảng trắng thừa ở đầu/cuối
job_title = " ".join(job_title_element.text.split())
print(job_title)
# Kết quả: "Data Analyst Ngân Hàng Shinhan"

Data Analyst Ngân Hàng Shinhan


In [93]:
company_name = new_soup.find('a', class_='name')
company_name = company_name.text
print(company_name)

Ngân Hàng TNHH MTV Shinhan Việt Nam


In [171]:
data = []

# Duyệt qua từng khung chứa thông tin
for item in new_soup.find_all('div', class_='list-info__content'):
    # Lấy element title và desc bên trong khung đó
    title_el = item.find('div', class_='list-info__content__title')
    desc_el = item.find('div', class_='list-info__content__desc')
    
    # Lấy văn bản và xóa khoảng trắng thừa (strip=True)
    title = title_el.get_text(strip=True) if title_el else ''
    desc = desc_el.get_text(strip=True) if desc_el else ''
    
    data.append({
        'title': title,
        'description': desc
    })

# In kết quả kiểm tra
for d in data:
    print(f"{d['title']}: {d['description']}")

Địa điểm: Hồ Chí Minh
Kinh nghiệm: 1 năm
Hạn ứng tuyển: 31/08/2026
